# 04 · AI Agents — The ReAct Loop

**Lecture slides 54–67.** An **AI agent** is an LLM running in a loop that can perceive (user prompts, tool results, memory), reason (break a goal into steps and choose the next one), and act (call tools, APIs, code, MCP servers).

| Paradigm | Flow |
|---|---|
| 1. Direct prompting | Question → LLM → Answer |
| 2. Tool-augmented | Question → LLM → Tool call → LLM → Answer |
| 3. **Autonomous agent** | Question → LLM → Tool → LLM → Tool → ••• → LLM → Answer |

**ReAct = Reason + Act.** At each step the model *reasons* about what to do, *acts* by calling a tool, *observes* the result, and repeats until it can answer. The only new code compared to notebook 03 is the `for` loop.

All live cells call `gemma4:e2b-mlx` through Ollama.

In [1]:
# %pip install -q ollama rank_bm25 numpy transformers torch ddgs
import ollama
from typing import Any, Callable, Dict, List, Tuple

NATIVE_MODEL_ID = "gemma4:e2b-mlx"

## Agent 1 · The thermostat agent (slide 66)

*"My teddy bear is cold. Please do something."* The agent must interpret the intent, **observe** the current temperature, **decide** on a better one, and **act** on the environment. The system prompt asks the model to state its reasoning before each tool call so we can watch the Reason → Act → Observe cycle.

In [2]:
# ── A tiny simulated environment + two tools ─────────────────────────────────
HOUSE = {"living room": 62, "bedroom": 65}  # current temperatures in °F


def get_room_temperature(room: str) -> str:
    temp = HOUSE.get(room.lower())
    return f"The {room} is currently {temp}°F." if temp is not None else f"Unknown room: {room}. Known rooms: {list(HOUSE)}"


def set_thermostat(room: str, temperature_f: float) -> str:
    if room.lower() not in HOUSE:
        return f"Unknown room: {room}. Known rooms: {list(HOUSE)}"
    HOUSE[room.lower()] = temperature_f
    return f"Thermostat in the {room} set to {temperature_f}°F."


THERMOSTAT_TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_room_temperature",
            "description": "Read the current temperature (°F) of a room in the house",
            "parameters": {
                "type": "object",
                "properties": {"room": {"type": "string", "description": "Room name, e.g. 'living room' or 'bedroom'"}},
                "required": ["room"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_thermostat",
            "description": "Set the target temperature (°F) of a room's thermostat",
            "parameters": {
                "type": "object",
                "properties": {
                    "room": {"type": "string", "description": "Room name"},
                    "temperature_f": {"type": "number", "description": "Target temperature in °F"},
                },
                "required": ["room", "temperature_f"],
            },
        },
    },
]
THERMOSTAT_TOOL_REGISTRY: Dict[str, Callable[..., str]] = {
    "get_room_temperature": get_room_temperature,
    "set_thermostat": set_thermostat,
}

REACT_SYSTEM_PROMPT = (
    "You are a smart-home agent. The teddy bear lives in the living room. "
    "Work step by step: before each tool call, write one short sentence explaining your reasoning. "
    "Check the current state before changing anything. When you are done, tell the user what you did."
)

In [3]:
# ── The ReAct loop: Reason -> Act -> Observe, until the model stops calling tools ─
def run_react_agent(user_message: str, tools_schema: List[Dict], registry: Dict[str, Callable[..., str]],
                    system_prompt: str, max_steps: int = 6, model: str = NATIVE_MODEL_ID) -> str:
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_message}]
    for step in range(1, max_steps + 1):
        response = ollama.chat(model=model, messages=messages, tools=tools_schema)
        messages.append(response.message)

        if response.message.content:
            print(f"[{step}] Thought:     {response.message.content.strip()}")
        if not response.message.tool_calls:
            return response.message.content  # no more actions — this is the final answer

        for tool_call in response.message.tool_calls:
            result = registry[tool_call.function.name](**tool_call.function.arguments)
            print(f"[{step}] Action:      {tool_call.function.name}({tool_call.function.arguments})")
            print(f"[{step}] Observation: {result}")
            messages.append({"role": "tool", "content": str(result), "tool_name": tool_call.function.name})

    return "Max steps reached without a final answer."


answer = run_react_agent(
    "My teddy bear is cold. Please do something.",
    THERMOSTAT_TOOLS_SCHEMA, THERMOSTAT_TOOL_REGISTRY, REACT_SYSTEM_PROMPT,
)
print(f"\nFinal answer: {answer}")
print("House state:", HOUSE)

[1] Thought:     My reasoning is to first check the current temperature in the living room to determine if an adjustment is necessary.
[1] Action:      get_room_temperature({'room': 'living room'})
[1] Observation: The living room is currently 62°F.


[2] Action:      set_thermostat({'room': 'living room', 'temperature_f': 72})
[2] Observation: Thermostat in the living room set to 72°F.


[3] Thought:     I first checked the temperature in the living room, which was 62°F, and then I set the thermostat to 72°F to make the room warmer.

Final answer: I first checked the temperature in the living room, which was 62°F, and then I set the thermostat to 72°F to make the room warmer.
House state: {'living room': 72, 'bedroom': 65}


**Try it:** ask *"It's too warm in the bedroom"* or *"Make every room 70°F"* — the second one needs several actions in a row.

## Agent 2 · Chaining several tools

The same loop with the weather and currency tools from notebook 03. A question that needs **two different tools** forces the agent to take more than one step before answering.

In [4]:
# ── Same two toy tools + JSON Schema definitions as notebook 03 ─────────────

def get_weather(city: str) -> str:
    """Toy weather lookup — a real implementation would call a weather API."""
    fake_weather = {"osaka": "28C, sunny", "tokyo": "26C, cloudy", "paris": "19C, rainy"}
    return fake_weather.get(city.lower(), f"No weather data for {city}")


def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Toy currency converter — a real implementation would call a live FX API."""
    fake_rates = {("USD", "JPY"): 147.5, ("JPY", "USD"): 1 / 147.5}
    rate = fake_rates.get((from_currency.upper(), to_currency.upper()))
    if rate is None:
        return f"No rate available for {from_currency} -> {to_currency}"
    return f"{amount * rate:.2f} {to_currency.upper()}"


# JSON Schema tool definitions — this is what the model actually sees
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "Convert an amount of money from one currency to another",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Amount to convert"},
                    "from_currency": {"type": "string", "description": "3-letter source currency code"},
                    "to_currency": {"type": "string", "description": "3-letter target currency code"},
                },
                "required": ["amount", "from_currency", "to_currency"],
            },
        },
    },
]

NATIVE_TOOL_REGISTRY: Dict[str, Callable[..., str]] = {
    "get_weather": get_weather,
    "convert_currency": convert_currency,
}

In [5]:
# ── Live Test 3: multi-step agent loop (model may chain several tool calls) ──
def run_agent(user_message: str, max_steps: int = 4, model: str = NATIVE_MODEL_ID) -> str:
    messages = [{"role": "user", "content": user_message}]
    for step in range(max_steps):
        response = ollama.chat(model=model, messages=messages, tools=TOOLS_SCHEMA)
        messages.append(response.message)
        print(f"\n[step {step + 1}] Model response: {response.message.content}")

        if not response.message.tool_calls:
            return response.message.content  # model is done — final answer

        for tool_call in response.message.tool_calls:
            fn = NATIVE_TOOL_REGISTRY[tool_call.function.name]
            result = fn(**tool_call.function.arguments)
            print(f"  [step {step + 1}] {tool_call.function.name}({tool_call.function.arguments}) -> {result}")
            messages.append({"role": "tool", "content": str(result), "tool_name": tool_call.function.name})
    return "Max steps reached without a final answer."

In [6]:
answer = run_agent("What's the weather in Tokyo, and how much is 50 USD in JPY?")
print(f"\nFinal answer: {answer}")


[step 1] Model response: 
  [step 1] get_weather({'city': 'Tokyo'}) -> 26C, cloudy
  [step 1] convert_currency({'amount': 50, 'from_currency': 'USD', 'to_currency': 'JPY'}) -> 7375.00 JPY



[step 2] Model response: The weather in Tokyo is 26°C and cloudy. 50 USD is equal to 7375.00 JPY.

Final answer: The weather in Tokyo is 26°C and cloudy. 50 USD is equal to 7375.00 JPY.


Notice that `run_agent` and `run_react_agent` are the same loop — only the tools and the system prompt change. That's the pattern behind every agent below.

## Agent 2b · Agentic RAG: retrieval as a tool

## Wiring hybrid search up as an agent tool

This is the actual RAG pattern: `retrieve_context` is declared with the same `tools=` JSON-Schema contract as Part 1, so the model decides *when* to search rather than always being handed context up front (compare the fixed retrieve → augment → generate pipeline in notebook 02). The system prompt asks it to search first, then answer only from what came back and cite results by their `[n]` marker.

In [7]:
# The pet knowledge base and hybrid (BM25 + embeddings + RRF) retriever built step by step in notebook 02
from rag_utils import PET_CORPUS, HybridRetriever

pet_retriever = HybridRetriever(PET_CORPUS)
CORPUS_BY_ID = pet_retriever.by_id
hybrid_search = pet_retriever.hybrid_search

/Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6346.70it/s]

In [8]:
# ── The retrieval tool itself, plus its JSON-Schema definition ────────────────
def retrieve_context(query: str, top_k: int = 3) -> str:
    """Tool: hybrid (lexical + vector) search over the pet knowledge base; returns citation-ready text blocks."""
    results = hybrid_search(query, top_k=top_k)
    if not results:
        return "No results found."
    return "\n".join(f"[{i}] {CORPUS_BY_ID[doc_id]}" for i, (doc_id, _score) in enumerate(results, start=1))


RAG_TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_context",
            "description": (
                "Hybrid lexical+vector search over a small pet knowledge base. "
                "Returns numbered, citation-ready text blocks most relevant to the query."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Natural-language search query"},
                    "top_k": {"type": "integer", "description": "Number of results to return (default 3)"},
                },
                "required": ["query"],
            },
        },
    }
]

RAG_TOOL_REGISTRY: Dict[str, Callable[..., str]] = {"retrieve_context": retrieve_context}

RAG_SYSTEM_PROMPT = (
    "You are a helpful assistant answering questions about pets. Always call retrieve_context "
    "to find relevant information before answering. Cite sources using the [n] markers from the "
    "retrieved text. If the retrieved context doesn't contain the answer, say so."
)


async def run_rag_agent(user_message: str, max_steps: int = 3, model: str = NATIVE_MODEL_ID) -> str:
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    for step in range(max_steps):
        response = ollama.chat(model=model, messages=messages, tools=RAG_TOOLS_SCHEMA)
        messages.append(response.message)

        if not response.message.tool_calls:
            return response.message.content  # model is done — final answer

        for tool_call in response.message.tool_calls:
            fn = RAG_TOOL_REGISTRY[tool_call.function.name]
            result = fn(**tool_call.function.arguments)
            print(f"  [step {step + 1}] {tool_call.function.name}({tool_call.function.arguments}) ->\n{result}")
            messages.append({"role": "tool", "content": result, "tool_name": tool_call.function.name})

    return "Max steps reached without a final answer."

In [9]:
# Needs facts from two different documents — one findable lexically (d5), one semantically (d4)
answer = await run_rag_agent(
    "Which pet can mimic human speech, and which pet is a hybrid with a wild leopard cat?"
)
print(f"\nFinal answer: {answer}")

  [step 1] retrieve_context({'query': 'pet that can mimic human speech'}) ->
[1] African Grey Parrots are known for their exceptional ability to mimic human speech and solve problems.
[2] Persian cats have long, thick fur and a calm, gentle temperament, making them popular indoor pets.
[3] Goldfish are freshwater fish commonly kept in home aquariums; they can live over 10 years with proper care.
  [step 1] retrieve_context({'query': 'pet hybrid with a wild leopard cat'}) ->
[1] Bengal cats are a hybrid breed resulting from crossing domestic cats with the Asian leopard cat.
[2] Persian cats have long, thick fur and a calm, gentle temperament, making them popular indoor pets.
[3] The Siberian Husky is a medium-sized working dog breed originally bred for sled pulling in cold climates.



Final answer: Based on the information retrieved:

1.  **Pet that can mimic human speech:** The **African Grey Parrot** is known for its exceptional ability to mimic human speech and solve problems [1].
2.  **Pet that is a hybrid with a wild leopard cat:** The **Bengal cat** is a hybrid breed resulting from crossing domestic cats with the Asian leopard cat [1].


## Agent 3 · A web search agent

RAG answers from a fixed, hand-curated knowledge base — great for a stable domain, but it goes stale the moment reality moves past its last update, and it only knows what you fed it. A **web search tool** trades that control for reach: instead of retrieving from documents you curated, the model queries the live web for whatever it needs *right now*. Same tool-calling shape as every other tool in this notebook — the "knowledge base" is just the entire internet, as fresh as the search engine's own index.

We use [`ddgs`](https://pypi.org/project/ddgs/), a free DuckDuckGo search wrapper that needs no API key or signup — a good fit for a notebook that otherwise runs entirely offline/local. The tradeoff versus RAG: no control over what's in the corpus, results vary run to run, and there's no guarantee the top hits are trustworthy — production agents typically pair this with a paid search API and stronger source vetting.

In [10]:
# %pip install -q ddgs

In [11]:
# ── The tool itself, plus its JSON-Schema definition ──────────────────────────
from ddgs import DDGS


def web_search(query: str, max_results: int = 5) -> str:
    """Tool: search the live web via DuckDuckGo; returns numbered, citation-ready text blocks."""
    results = DDGS().text(query, max_results=max_results)
    if not results:
        return "No results found."
    return "\n".join(
        f"[{i}] {r['title']}\n    {r['href']}\n    {r['body']}" for i, r in enumerate(results, start=1)
    )


WEB_TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": (
                "Search the live web via DuckDuckGo. Returns numbered, citation-ready text "
                "blocks (title, URL, snippet) for the top results."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"},
                    "max_results": {"type": "integer", "description": "Number of results to return (default 5)"},
                },
                "required": ["query"],
            },
        },
    }
]

WEB_TOOL_REGISTRY: Dict[str, Callable[..., str]] = {"web_search": web_search}

# Sanity check: call the tool directly, no model involved yet
print(web_search("Ollama latest release version", max_results=3))

[1] Releases · ollama/ollama
    https://github.com/ollama/ollama/releases
    Releases · ollama/ollama. Release list.github-actions released this 15 Sep 21:21.
[2] Ollama Latest Version (v0.33.2) + Version History
    https://localaimaster.com/blog/ollama-version-history
    Published on April 10, 2026 · Updated August 30, 2026 — 18 min read. The latest Ollama version is v0.33.2, released August 27, 2026. Run ollama --version to see what you have — if it is anything below 0.32, you are missing the new interactive agent mode...
[3] ollama/ollama on GitHub | Release Alert
    https://releasealert.dev/github/ollama/ollama
    Latest releases for ollama/ollama on GitHub. Latest version: v0.34.3-rc1, last published: September 19, 2026.


## Wiring it into an agent loop

Same shape as `run_rag_agent` above — swap the tool, keep the loop. The test question below is deliberately something the model can't know from training alone and isn't in the pet corpus: it can only be answered by actually searching.

In [12]:
WEB_SYSTEM_PROMPT = (
    "You are a helpful assistant with access to a live web search tool. Call web_search "
    "whenever the answer depends on current or fast-changing information you might not know. "
    "Cite sources using the [n] markers from the search results."
)


async def run_web_agent(user_message: str, max_steps: int = 3, model: str = NATIVE_MODEL_ID) -> str:
    messages = [
        {"role": "system", "content": WEB_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    for step in range(max_steps):
        response = ollama.chat(model=model, messages=messages, tools=WEB_TOOLS_SCHEMA)
        messages.append(response.message)

        if not response.message.tool_calls:
            return response.message.content  # model is done — final answer

        for tool_call in response.message.tool_calls:
            fn = WEB_TOOL_REGISTRY[tool_call.function.name]
            result = fn(**tool_call.function.arguments)
            print(f"  [step {step + 1}] {tool_call.function.name}({tool_call.function.arguments}) ->\n{result}")
            messages.append({"role": "tool", "content": result, "tool_name": tool_call.function.name})

    return "Max steps reached without a final answer."

In [13]:
answer = await run_web_agent("What is the most recent version of Ollama, and when was it released?")
print(f"\nFinal answer: {answer}")

  [step 1] web_search({'query': 'most recent version of Ollama and release date'}) ->
[1] Ollama - Wikipedia
    https://en.wikipedia.org/wiki/Ollama
    Ollama is an open-source software platform developed by Jeffrey Morgan and Michael Chiang in 2023 for running and managing large language models on local GPU infrastructure and through hosted cloud models.
[2] Releases · ollama/ollama - GitHub
    https://github.com/ollama/ollama/releases
    Get up and running with Kimi, GLM, MiniMax, DeepSeek, gpt-oss, Qwen, Gemma and other models. - ollama/ollama
[3] Ollama Latest Version 2026: v0.33.1 + 10 Best Models
    https://www.promptquorum.com/local-llms/top-open-source-models-ollama
    The current Ollama version is v0.33.1 (August 26, 2026); v0.33.2 is available as a release candidate. The newest models added recently are Laguna XS 2.1 (Poolside, 33B/3B active MoE, agentic coding) and Kimi K2.7 Code (Moonshot AI, coding-focused, built on K2.6).
[4] Ollama Release Notes - September 2026 La


Final answer: Based on the search results, one source indicates that the current Ollama version is **v0.33.1**, released around **August 26, 2026** [3].

Please note that version information can change rapidly, so for the absolute latest official version, it is best to check the official Ollama GitHub releases page or documentation.


## Key takeaways

- An **agent** is an LLM in a loop: **Reason** (decide the next step) → **Act** (call a tool) → **Observe** (read the result) → repeat until done.
- The loop is tiny; what makes each agent different is its **tools** and its **system prompt**. Swap the tools and you get a smart-home agent, a RAG agent or a web research agent.
- Always cap the loop (`max_steps`) — models can loop forever or call the wrong tool, and the only guardrails are the ones you write.
- Agents can also be started by **external triggers** (webhooks, cron jobs, events) instead of a human prompt, and your favorite coding assistant is exactly this loop with file, shell and search tools.